In [4]:
import os
import glob
import re
import ast
import pandas as pd
import pickle as pkl

In [2]:
# ---- SETTINGS ----
# Folder containing your corners_*.txt files
FOLDER = "./corners"   # <-- change this if your files are elsewhere

# Regex to extract M, N, seed from filenames like corners_1_3_1.txt
fname_pattern = re.compile(r"corners_(\d+)_(\d+)_(\d+)\.txt$")


def parse_corners_file(path):
    """
    Parse one corners_*.txt file.
    Returns (cornerpairs_M, cornerpairs_N), where each is a list of pairs.
    Each pair is [(x1, y1), (x2, y2)].
    """
    cornerpairs_M = None
    cornerpairs_N = None

    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            # Skip junk lines like "beep"
            if line.startswith("M "):
                # Everything after the first space should be a Python-like list
                data_str = line.split(" ", 1)[1]
                cornerpairs_M = ast.literal_eval(data_str)
            elif line.startswith("N "):
                data_str = line.split(" ", 1)[1]
                cornerpairs_N = ast.literal_eval(data_str)

    return cornerpairs_M, cornerpairs_N


rows = []

# Loop over all txt files in the folder
for path in glob.glob(os.path.join(FOLDER, "corners_*.txt")):
    filename = os.path.basename(path)
    m = fname_pattern.match(filename)
    if not m:
        # Skip files that don't match the naming pattern
        continue

    M_val = int(m.group(1))
    N_val = int(m.group(2))
    seed_val = int(m.group(3))

    cornerpairs_M, cornerpairs_N = parse_corners_file(path)

    rows.append({
        "M": M_val,
        "N": N_val,
        "seed": seed_val,
        "cornerpairs_M": cornerpairs_M,  # list of pairs for M
        "cornerpairs_N": cornerpairs_N,  # list of pairs for N
    })

# Build the dataframe
df = pd.DataFrame(rows)

# Sort rows if you like (optional)
df = df.sort_values(["M", "N", "seed"]).reset_index(drop=True)

df


,M,N,seed,cornerpairs_M,cornerpairs_N
0,1,1,1,"[[(9.773894590841444, 11.830850267032698), (7....","[[(12.536476558798045, 6.491506018575801), (1...."
1,1,1,2,"[[(10.845181218561988, 14.92229344424614), (6....","[[(6.973407931459682, 4.776976917805161), (0.5..."
2,1,1,3,"[[(14.934672532656942, 7.053952612836719), (3....","[[(11.37345369430226, 8.866493743969764), (10...."
3,1,1,4,"[[(13.769325646315783, 12.006785272437128), (0...","[[(10.97842063318077, 12.819725369870207), (4...."
4,1,1,5,"[[(11.92790348348545, 14.136754256655754), (9....","[[(8.156412888538956, 8.609117818921511), (7.0..."
...,...,...,...,...,...
295,5,1,16,"[[(9.645335283756154, 14.22575316160669), (5.2...","[[(14.898658743500246, 4.574515715668221), (1...."
296,5,1,17,"[[(12.694917136926362, 14.17790871757334), (7....","[[(8.872634555409588, 7.779544305672322), (4.5..."
297,5,1,18,"[[(10.37820150531828, 4.866572155293737), (3.9...","[[(9.156745362099949, 7.079767570139473), (6.3..."
298,5,1,19,"[[(4.6659760575213225, 8.538600238144792), (1....","[[(10.8755464931482, 14.331030619073498), (3.9..."


In [7]:
with open('all_corner_points.pkl', 'wb') as f:
    pkl.dump(df, f)
df.to_csv('all_corner_points.csv', index=False)